# Constitutional AI — Dev Log

## Objetivo e papel no pipeline

`core/constitutional_ai` é o primeiro módulo da **Onda 2 do V2**: uma
constituição declarativa (`constitution.yaml`) de princípios "linha
vermelha" compilada em checks executáveis — versão simples, conforme o
próprio ROADMAP pede ("regras declarativas → políticas executáveis, versão
simples").

**Diferença para o `policy_engine` (V1)**: `policy_engine` decide o que é
**permitido com que condições** (branching de múltiplos outcomes por
política); `constitutional_ai` define o que **nunca pode acontecer**,
independente de qual política se aplique — dois níveis de governança
complementares, não concorrentes.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.constitutional_ai.engine import check_constitution

# Cenário: decisão automatizada de alto impacto, sem via de revisão humana
# disponível -- viola CONST-03 (Supervisão humana), mesmo que todos os outros
# princípios estejam OK.
context = {
    "automated_decision": True,
    "explainable": True,
    "high_impact": True,
    "human_review_available": False,
    "in_production": True,
    "audit_logging_enabled": True,
    "prompt_security_scanned": True,
}
result = check_constitution(context)
print("Compliant?", result.compliant)
for v in result.violations:
    print(f" - {v.principle_id} ({v.principle}) | severidade={v.severity.value}")
    print(f"   {v.rationale}")
print()
print(result.summary)

Compliant? False
 - CONST-03 (Supervisão humana) | severidade=critical
   Decisões automatizadas de alto impacto (ex. crédito, emprego, acesso a benefícios) exigem via de recurso e revisão humana disponível (LGPD Art. 20, §1º).

1 violação(ões) constitucional(is) detectada(s) de 6 artigo(s) avaliado(s): CONST-03 (Supervisão humana).


## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/constitutional_ai/tests -v
```

10 testes: contexto conforme; cada um dos 6 princípios violado
individualmente; múltiplas violações simultâneas; contexto vazio;
constituição customizada via `constitution_path` (prova que o motor é
genérico).

## Handoff Summary

- **Status:** ✅ done — 10/10 testes passando.
- **TODO onda futura:** compor `constitutional_ai` + `policy_engine` num
  veredito único (`governance_copilot`); operadores de condição mais ricos
  (`_any`/`_all`, como no `policy_engine`) se necessário.